# WS1: Rating Prediction (Regression) & Empfehlung (Klassifikation)

**Ansatz A** – Wie viele Sterne würde ein User einem Buch geben? (Regression)  
**Ansatz B** – Würde ein User ein Buch empfehlen, ja oder nein? (binäre Klassifikation)

Arbeitsebene: **Reviewebene** – jede Zeile = eine Bewertung eines Users zu einem Buch.

Dieses Notebook deckt **nur Schritt 1 (Laden & Joinen) und Schritt 2 (Minimal-Bereinigung + Subsampling)** ab.

---
## 0. Imports

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

RANDOM_STATE = 42
print("OK")

OK


---
## Schritt 1: Daten laden & joinen

Wir haben zwei CSV-Dateien:
- `Books_rating.csv` – eine Zeile pro Review (User, Buch, Score, Text, ...)
- `books_data.csv` – eine Zeile pro Buch (Metadaten: Genre, Autor, Verlag, ...)

Join über `Title` (inner join = nur Bücher, für die beides vorhanden ist).

In [2]:
df_books   = pd.read_csv('../data/books_data.csv')
df_ratings = pd.read_csv('../data/Books_rating.csv')

print(f"books_data:    {df_books.shape}")
print(f"Books_rating:  {df_ratings.shape}")

df = pd.merge(df_books, df_ratings, on='Title', how='inner')
print(f"\nNach Join:     {df.shape}")
df.head(3)

books_data:    (212404, 10)
Books_rating:  (3000000, 10)

Nach Join:     (3000000, 19)


,Title,description,authors,image,previewLink,publisher,publishedDate,infoLink,categories,ratingsCount,Id,Price,User_id,profileName,review/helpfulness,review/score,review/time,review/summary,review/text
0,Its Only Art If Its Well Hung!,NaN,['Julie Strain'],http://books.google.com/books/content?id=DykPA...,http://books.google.nl/books?id=DykPAAAACAAJ&d...,NaN,1996,http://books.google.nl/books?id=DykPAAAACAAJ&d...,['Comics & Graphic Novels'],NaN,1882931173,NaN,AVCGYZL8FQQTD,"Jim of Oz ""jim-of-oz""",7/7,4.0,940636800,Nice collection of Julie Strain images,This is only for Julie Strain fans. It's a col...
1,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,['Philip Nel'],http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,A&C Black,2005-01-01,http://books.google.nl/books?id=IjvHQsCn_pgC&d...,['Biography & Autobiography'],NaN,0826414346,NaN,A30TK6U7DNS82R,Kevin Killian,10/10,5.0,1095724800,Really Enjoyed It,I don't care much for Dr. Seuss but after read...
2,Dr. Seuss: American Icon,Philip Nel takes a fascinating look into the k...,['Philip Nel'],http://books.google.com/books/content?id=IjvHQ...,http://books.google.nl/books?id=IjvHQsCn_pgC&p...,A&C Black,2005-01-01,http://books.google.nl/books?id=IjvHQsCn_pgC&d...,['Biography & Autobiography'],NaN,0826414346,NaN,A3UH4UZ4RSVO82,John Granger,10/11,5.0,1078790400,Essential for every personal and Public Library,"If people become the books they read and if ""t..."


In [3]:
# Überblick: Wie viele Werte fehlen pro Spalte?
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
pd.DataFrame({'fehlend_absolut': missing, 'fehlend_%': missing_pct}).sort_values('fehlend_%', ascending=False)

,fehlend_absolut,fehlend_%
Price,2518829,84.0
ratingsCount,1360694,45.4
publisher,782617,26.1
description,640225,21.3
User_id,561787,18.7
profileName,561905,18.7
categories,551498,18.4
image,540306,18.0
authors,390634,13.0
publishedDate,354581,11.8


---
## Schritt 2: Minimal-Bereinigung VOR dem Subsampling

### Warum erst bereinigen, dann sampeln?

Das Sampling stratifiziert nach `review/score`. Wenn noch Zeilen ohne Score drin sind, können sie nicht zugeordnet werden – das macht den Code zwar nicht falsch, aber es ist sauberer, das explizit vorher zu erledigen. Außerdem: wenn Zeilen ohne `User_id` erst *nach* dem Sampling rausfallen, kann die Klassenverteilung im Sample leicht kippen.

---

### Entscheidung 1: Welche Spalten droppen wir?

Wir droppen nur Spalten, bei denen **zwei Bedingungen** gleichzeitig zutreffen:
1. Sie können nicht sinnvoll als Feature genutzt werden
2. Eine Imputation wäre bei so vielen Fehlenden mehr Erfindung als Daten

| Spalte | Fehlend % | Entscheidung | Begründung |
|---|---|---|---|
| `Price` | ~84 % | **Droppen** | Bei 84 % Fehlenden wäre jede Imputation unrealistisch. Außerdem ist der Preis kein starker Prädiktor für persönliche Bewertungen – ein teures Buch bekommt nicht automatisch mehr Sterne. |
| `ratingsCount` | ~45 % | **Droppen** | Kommt aus der Google Books API, nicht aus unserem Datensatz – wir wissen nicht wie er berechnet wird. Wir berechnen `book_review_count` selbst aus unseren Daten (100 % vorhanden, keine Fehlenden). |
| `description` | ~21 % | **Droppen** | Freitext. Ohne NLP/Textverarbeitung (nicht im Kurs) nicht als Feature nutzbar. |
| `image` | ~16 % | **Droppen** | URL zu einem Bild. Kein Feature. |
| `previewLink` | ~11 % | **Droppen** | URL. Kein Feature. |
| `infoLink` | ~11 % | **Droppen** | URL. Kein Feature. |
| `profileName` | ~19 % | **Droppen** | Wir verwenden `User_id` als Nutzer-Identifier, nicht den Klarnamen. Die beiden sind redundant, `User_id` ist vollständiger. |

**Was wir NICHT droppen** (auch wenn Werte fehlen):

| Spalte | Fehlend % | Entscheidung | Begründung |
|---|---|---|---|
| `categories` | ~18 % | **Behalten** | Genre ist ein inhaltlich sinnvolles Feature. Fehlende füllen wir später mit "Unknown". |
| `authors` | ~13 % | **Behalten (vorerst)** | Könnte interessant sein; entscheiden wir im Preprocessing-Schritt. |
| `publisher` | ~26 % | **Behalten (vorerst)** | Wie `authors` – entscheiden wir im Preprocessing-Schritt. |
| `publishedDate` | ~12 % | **Behalten (vorerst)** | Für `book_age` als Feature nutzbar. Zeilen ohne Datum bekommen dieses Feature einfach nicht. |

In [ ]:
cols_to_drop = ['Price', 'ratingsCount', 'description', 'image', 'previewLink', 'infoLink', 'profileName']
df = df.drop(columns=cols_to_drop)

print(f"Spalten nach Drop: {list(df.columns)}")
print(f"Shape: {df.shape}")

---
### Entscheidung 2: Welche ZEILEN droppen wir?

#### 2a: Zeilen ohne `review/score`

`review/score` ist unsere **Zielvariable** – ohne sie ist eine Zeile für uns komplett wertlos. Weder für Training noch für Evaluation nutzbar. Diese Zeilen fliegen raus. (Betrifft laut Exploration fast keine Zeilen, ~0 %.)

#### 2b: Zeilen ohne `User_id` – sinnvoll?

Das ist die interessantere Frage. Es gibt zwei Argumente:

**Für das Droppen:**
- Ohne `User_id` können wir keine user-bezogenen Features berechnen: `user_avg_score`, `user_review_count`, `user_score_std`, `user_positive_ratio`
- Diese User-Features sind wahrscheinlich die stärksten Prädiktoren (manche User geben immer 5 Sterne, andere immer 2)
- Eine Zeile ohne User_id hätte für all diese Features NaN – das Modell kann sie nicht sinnvoll nutzen

**Gegen das Droppen (Vorsicht):**
- ~19 % der Zeilen haben keine User_id – das ist nicht wenig
- Diese Zeilen enthalten trotzdem einen Score und Buchmetadaten
- Man könnte sie mit nur buch-bezogenen Features trainieren

**Unsere Entscheidung: Droppen.**  
Der Kern unseres Ansatzes ist personalisierte Empfehlung – die setzt einen identifizierbaren User voraus. Eine Zeile ohne User_id ist ein anonymes Review; für unser Ziel nicht verwertbar. Wir verlieren ~19 % der Daten, aber bei 2,4 Mio Zeilen bleibt genug übrig.

In [4]:
print(f"Vor Zeilenbereinigung: {df.shape}")

df = df.dropna(subset=['review/score', 'User_id'])

print(f"Nach Zeilenbereinigung: {df.shape}")
print(f"Weggefallen: {3_000_000 - len(df):,} Zeilen")

Vor Zeilenbereinigung: (3000000, 19)
Nach Zeilenbereinigung: (2438213, 19)
Weggefallen: 561,787 Zeilen


In [5]:
# Score zu int casten – kann im CSV als 1.0, 2.0 ... vorliegen
# Wenn wir das nicht tun, kann der groupby beim Sampling Probleme machen
df['review/score'] = df['review/score'].astype(int)

print("Score-Verteilung im bereinigten Datensatz:")
print(df['review/score'].value_counts().sort_index())
print()
print("Anteile:")
print(df['review/score'].value_counts(normalize=True).sort_index().round(3))

Score-Verteilung im bereinigten Datensatz:
review/score
1     151158
2     119634
3     212916
4     492174
5    1462331
Name: count, dtype: int64

Anteile:
review/score
1    0.062
2    0.049
3    0.087
4    0.202
5    0.600
Name: proportion, dtype: float64


---
### Entscheidung 3: Wie sampeln wir?

#### Warum nicht einfach die ersten N Zeilen nehmen?
Die Daten sind zeitlich sortiert. Die ersten 60.000 Zeilen kämen alle aus demselben Zeitraum oder derselben Kategorie. Das Sample wäre nicht repräsentativ.

#### Warum nicht nur vollständige Zeilen?
Nach unseren Drops oben sind noch ~19 % der Daten mit einzelnen fehlenden Werten übrig (v.a. in `categories`, `authors`, `publisher`). Nur vollständige Zeilen zu nehmen würde:
1. Den Sample auf ~5-10 % der Daten reduzieren
2. Systematisch Bücher ohne vollständige Metadaten ausschließen (der "Long Tail" wäre weg)
3. User-Stats aus sehr wenigen Reviews berechnen → instabil

Diese fehlenden Werte behandeln wir im **Preprocessing-Schritt** (z.B. `categories` → "Unknown").

#### Stratified Sampling nach `review/score`
Wir sampeln innerhalb jeder Score-Klasse (1–5) separat mit dem gleichen Anteil `frac`. So bleibt die Score-Verteilung im Sample identisch zum Original.

**`frac = ?`** – das ist deine Entscheidung. Zur Orientierung:
- `frac=0.02` → ~48.000 Zeilen (knapp, user/book-Aggregate aus wenig Reviews)
- `frac=0.03` → ~72.000 Zeilen (robuster, noch schnell zu rechnen)
- `frac=0.05` → ~120.000 Zeilen (sehr robust, GridSearchCV dauert länger)

Setze `SAMPLE_FRAC` auf deinen Wert und führe die Zelle aus.

In [ ]:
SAMPLE_FRAC = 0.03  # <-- hier anpassen

# pd.concat statt groupby().apply(), weil pandas 3.x die Groupby-Spalte
# aus dem apply()-Ergebnis entfernt – dann wäre review/score weg
df_sample = pd.concat([
    group.sample(frac=SAMPLE_FRAC, random_state=RANDOM_STATE)
    for _, group in df.groupby('review/score')
]).reset_index(drop=True)

print(f"Sample-Größe: {df_sample.shape[0]:,} Zeilen, {df_sample.shape[1]} Spalten")
print(f"Spalten: {list(df_sample.columns)}")

In [7]:
# Prüfen: Score-Verteilung Original vs. Sample – müssen übereinstimmen
orig_dist   = df['review/score'].value_counts(normalize=True).sort_index().round(3)
sample_dist = df_sample['review/score'].value_counts(normalize=True).sort_index().round(3)

check = pd.DataFrame({'Original': orig_dist, 'Sample': sample_dist})
check['Differenz'] = (check['Original'] - check['Sample']).abs()
print(check)
print(f"\nMax. Abweichung: {check['Differenz'].max():.4f} (sollte ~0 sein)")

KeyError: 'review/score'

In [ ]:
# Visualisierung: Score-Verteilung Original vs. Sample
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

orig_dist.plot(kind='bar', ax=axes[0], color='steelblue')
axes[0].set_title(f'Original (n={len(df):,})')
axes[0].set_xlabel('Score')
axes[0].set_ylabel('Anteil')
axes[0].tick_params(axis='x', rotation=0)

sample_dist.plot(kind='bar', ax=axes[1], color='darkorange')
axes[1].set_title(f'Sample (n={len(df_sample):,})')
axes[1].set_xlabel('Score')
axes[1].set_ylabel('Anteil')
axes[1].tick_params(axis='x', rotation=0)

plt.suptitle('Score-Verteilung: Original vs. Sample (sollten identisch aussehen)')
plt.tight_layout()
plt.show()

In [ ]:
# Abschließender Überblick: Was haben wir nach dem Sampling?
print("Verbleibende Spalten und ihre fehlenden Werte:")
missing = df_sample.isnull().sum()
missing_pct = (missing / len(df_sample) * 100).round(1)
pd.DataFrame({'fehlend_absolut': missing, 'fehlend_%': missing_pct})